# バッチ処理と GPU で推論を高速化する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_transformers/batch_inference.ipynb)

NLP クックブック 12-08 ／ Transformers の基本操作

数万件のテキストに対してモデルの推論を回したい。1 件ずつ処理すると遅すぎる。

In [1]:
import time
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
name = "lxyuan/distilbert-base-multilingual-cased-sentiments-student"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name).to(device).eval()

base = ["とても良かった。", "最悪だった。二度と行かない。", "普通でした。可もなく不可もなく、値段相応という印象です。" * 4]
texts = [base[2] if i % 8 == 0 else base[i % 2] for i in range(96)]   # 8 件に 1 件だけ長い文

@torch.inference_mode()
def predict(texts, batch_size, sort_by_length=False):
    order = sorted(range(len(texts)), key=lambda i: len(texts[i])) if sort_by_length else list(range(len(texts)))
    preds = [None] * len(texts)
    for i in range(0, len(order), batch_size):
        idx = order[i:i + batch_size]
        batch = tokenizer([texts[j] for j in idx], padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)
        for j, p in zip(idx, model(**batch).logits.argmax(-1).tolist()):
            preds[j] = model.config.id2label[p]
    return preds

print("device:", device)
for label, kwargs in [("1 件ずつ", dict(batch_size=1)), ("バッチ 32", dict(batch_size=32)), ("バッチ 32 + 長さでソート", dict(batch_size=32, sort_by_length=True))]:
    t = time.perf_counter()
    preds = predict(texts, **kwargs)
    print(f"{label:22} {time.perf_counter() - t:5.2f} 秒  {preds[:3]}")

device: cpu
1 件ずつ                  36.06 秒  ['neutral', 'negative', 'positive']
バッチ 32                 91.19 秒  ['neutral', 'negative', 'positive']
バッチ 32 + 長さでソート         6.40 秒  ['neutral', 'negative', 'positive']


- `torch.inference_mode()` (または `no_grad()`) で勾配計算を止める。忘れるとメモリを浪費し、遅くなる。
- 長さの近い文を同じバッチにまとめると、パディングが減って速くなる。結果は元の順番に戻す。
- GPU ではバッチ化の効果が非常に大きい (数十倍)。さらに `model.half()` (半精度) で速度とメモリが改善する。
- `pipeline(..., device=0, batch_size=32)` でも同様のバッチ処理ができる。